In [2]:
import os
os.chdir("C:/Users/Lenovo/Desktop/Agai Project")  # your actual project path
print(os.getcwd())

C:\Users\Lenovo\Desktop\Agai Project


In [4]:
print(os.path.exists("data/processed/text/train.csv"))

True


In [6]:
import numpy as np 
import pandas as pd
train_df = pd.read_csv("data/processed/text/train.csv")
question_embeddings = np.load("data/processed/text/train_question_embeddings.npy")

print(train_df.shape)
print(question_embeddings.shape)

(24332, 12)
(24332, 384)


In [8]:
import sys
!{sys.executable} -m pip install faiss-cpu

  Using cached faiss_cpu-1.15.0-cp313-cp313-win_amd64.whl.metadata (7.8 kB)
Using cached faiss_cpu-1.15.0-cp313-cp313-win_amd64.whl (16.3 MB)



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
import faiss

embedding_dim = question_embeddings.shape[1]  # 384

index = faiss.IndexFlatL2(embedding_dim)
index.add(question_embeddings.astype('float32'))

print("Number of vectors in index:", index.ntotal)

Number of vectors in index: 24332


In [10]:
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer('all-MiniLM-L6-v2')

def retrieve_similar_questions(user_question, top_k=3):
    query_embedding = embed_model.encode([user_question]).astype('float32')
    distances, indices = index.search(query_embedding, top_k)
    
    results = train_df.iloc[indices[0]][['question', 'answer', 'category', 'role', 'difficulty']].copy()
    results['distance'] = distances[0]
    return results

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [11]:
test_question = "Tell me about a time you had to work under pressure"
results = retrieve_similar_questions(test_question)
print(results)

                                             question  \
8   Tell me about a time you worked well within a ...   
50  Tell me about a time you worked well within a ...   
97  Tell me about a time you worked well within a ...   

                                               answer            category  \
8   I believe communication and mutual respect are...  Team Collaboration   
50  I believe communication and mutual respect are...  Team Collaboration   
97  I believe communication and mutual respect are...  Team Collaboration   

                   role difficulty  distance  
8       DevOps Engineer       Easy  0.974112  
50      Product Manager     Medium  0.974112  
97  Marketing Associate       Hard  0.974112  


In [15]:
def retrieve_similar_questions(user_question, top_k=3, domain=None):
    query_embedding = embed_model.encode([user_question]).astype('float32')
    
    if domain:
        mask = train_df['domain'] == domain
        filtered_df = train_df[mask].reset_index(drop=True)
        filtered_embeddings = question_embeddings[mask.values]
        temp_index = faiss.IndexFlatL2(embedding_dim)
        temp_index.add(filtered_embeddings.astype('float32'))
        distances, indices = temp_index.search(query_embedding, top_k)
        results = filtered_df.iloc[indices[0]][['question', 'answer', 'category', 'role', 'difficulty']].copy()
    else:
        distances, indices = index.search(query_embedding, top_k)
        results = train_df.iloc[indices[0]][['question', 'answer', 'category', 'role', 'difficulty']].copy()
    
    results['distance'] = distances[0]
    return results

In [19]:
import os
print(os.path.exists(".env"))
print(os.getcwd())

False
C:\Users\Lenovo\Desktop\Agai Project


In [21]:
with open(".env", "w") as f:
    f.write("ANTHROPIC_API_KEY=your_actual_key_here")

print("Created .env — now edit it with your real key")

Created .env — now edit it with your real key


In [22]:
with open(".gitignore", "r") as f:
    print(f.read())

data/*
!data/README.md
.env



In [28]:
import sys
!{sys.executable} -m pip install google-generativeai

  Using cached google_generativeai-0.8.6-py3-none-any.whl.metadata (3.9 kB)
  Using cached google_ai_generativelanguage-0.6.15-py3-none-any.whl.metadata (5.7 kB)
  Using cached google_api_core-2.34.0-py3-none-any.whl.metadata (2.9 kB)
  Using cached google_api_python_client-2.198.0-py3-none-any.whl.metadata (7.0 kB)
  Using cached google_auth-2.56.3-py3-none-any.whl.metadata (6.0 kB)
  Using cached protobuf-7.35.1-cp310-abi3-win_amd64.whl.metadata (595 bytes)
  Using cached proto_plus-1.28.3-py3-none-any.whl.metadata (2.2 kB)
  Using cached protobuf-5.29.6-cp310-abi3-win_amd64.whl.metadata (592 bytes)
  Using cached googleapis_common_protos-1.75.1-py3-none-any.whl.metadata (8.5 kB)
INFO: pip is looking at multiple versions of google-api-core to determine which version is compatible with other requirements. This could take a while.
  Using cached google_api_core-2.33.0-py3-none-any.whl.metadata (3.2 kB)
  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
INFO: pip is looki


[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [30]:
import google.generativeai as genai
from dotenv import load_dotenv
import os

load_dotenv()
genai.configure(api_key=os.getenv("GEMINI_API_KEY"))
model_gemini = genai.GenerativeModel("gemini-2.0-flash")

print("Gemini configured successfully")

Gemini configured successfully


In [36]:
import google.generativeai as genai
from dotenv import load_dotenv
import os

load_dotenv()
genai.configure(api_key=os.getenv("GEMINI_API_KEY"))
model_gemini = genai.GenerativeModel("gemini-flash-latest")

def evaluate_answer(question, user_answer, domain="HR"):
    retrieved = retrieve_similar_questions(question, top_k=2, domain=domain)
    context = "\n\n".join([
        f"Reference Question: {row['question']}\nIdeal Answer: {row['answer']}"
        for _, row in retrieved.iterrows()
    ])
    
    prompt = f"""You are an expert interview coach evaluating a candidate's answer.

QUESTION ASKED: {question}

CANDIDATE'S ANSWER: {user_answer}

REFERENCE MATERIAL (similar questions with ideal answers, for grounding your judgment):
{context}

Evaluate the candidate's answer and respond ONLY in this exact JSON format, no other text, no markdown code blocks:
{{
  "content_score": <integer 1-10>,
  "strengths": "<1-2 sentences on what was good>",
  "missing_points": "<1-2 sentences on what was missing or could improve>",
  "structure_feedback": "<1 sentence on clarity/structure of the answer>"
}}"""

    response = model_gemini.generate_content(prompt)
    return response.text.strip()

In [37]:
for m in genai.list_models():
    if 'generateContent' in m.supported_generation_methods:
        print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-3.6-flash
models/gemini-3.7-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.6-preview
models/gemini-robotics-er-2-preview
models/gemini-2.5-computer-use-p

In [38]:
import json

test_question = "Tell me about a time you had to work under pressure"
test_answer = "Once I had a project deadline that got moved up. I stayed organized by making a task list and communicating with my team about priorities. We finished on time."

result = evaluate_answer(test_question, test_answer, domain="HR")
print(result)

# Gemini sometimes wraps JSON in markdown code blocks — strip if needed
clean_result = result.replace("```json", "").replace("```", "").strip()
parsed = json.loads(clean_result)
print(parsed)

{
  "content_score": 5,
  "strengths": "You clearly identified practical actions taken to handle the situation, such as prioritizing tasks and maintaining proactive team communication.",
  "missing_points": "The response lacks specific context, such as how much the deadline shifted, the stakes involved, and measurable results beyond simply finishing on time.",
  "structure_feedback": "While the response follows a basic STAR progression, it is too brief and needs more elaboration to make a strong impact."
}
{'content_score': 5, 'strengths': 'You clearly identified practical actions taken to handle the situation, such as prioritizing tasks and maintaining proactive team communication.', 'missing_points': 'The response lacks specific context, such as how much the deadline shifted, the stakes involved, and measurable results beyond simply finishing on time.', 'structure_feedback': 'While the response follows a basic STAR progression, it is too brief and needs more elaboration to make a str

In [39]:
test_q1 = "Explain how a hash table works"
test_a1 = "A hash table stores key-value pairs. It uses a hash function to convert the key into an index in an array, so you can access values in constant time on average. If two keys hash to the same index, that's called a collision, and it's usually handled with chaining or open addressing."

result1 = evaluate_answer(test_q1, test_a1, domain="Technical")
print(result1)
print()

{
  "content_score": 8,
  "strengths": "The candidate provides a concise and accurate explanation of core concepts, including key-value storage, hash function mapping, average O(1) time complexity, and collision resolution techniques.",
  "missing_points": "The answer could be improved by mentioning load factors, dynamic resizing, and worst-case O(n) time complexity.",
  "structure_feedback": "The explanation is logically sequenced and directly answers the question without unnecessary fluff."
}



In [40]:
test_q2 = "Explain how a hash table works"
test_a2 = "It's like a data structure that stores stuff and you can find things fast in it."

result2 = evaluate_answer(test_q2, test_a2, domain="Technical")
print(result2)
print()

{
  "content_score": 2,
  "strengths": "You correctly identified that a hash table is a data structure optimized for fast data lookup.",
  "missing_points": "The answer completely misses key technical concepts such as key-value pairs, the role of a hash function, bucket indexing, collision resolution techniques (like chaining or open addressing), and time complexity.",
  "structure_feedback": "The response is overly brief and informal; aim for a structured technical explanation covering the core mechanism, workflow, and collision handling."
}



In [41]:
test_q3 = "Tell me about a time you showed leadership"
test_a3 = "I really like pizza and my favorite color is blue."

result3 = evaluate_answer(test_q3, test_a3, domain="HR")
print(result3)

{
  "content_score": 1,
  "strengths": "The sentence is grammatically correct and concise.",
  "missing_points": "The response is entirely off-topic and fails to mention any leadership experience, challenges faced, actions taken, or results achieved.",
  "structure_feedback": "The answer completely lacks a behavioral structure such as the STAR method and fails to answer the question asked."
}


In [42]:
test_q4 = "Tell me about a time you showed leadership"
test_a4 = "During my final year project, our team was behind schedule with three weeks left. I proposed we split into two sub-teams and I coordinated daily 15-minute standups to track blockers. I also personally took on the hardest unassigned module. We ended up submitting two days early, and two teammates told me the structure I set up helped them stay focused. It taught me that leadership isn't about doing everything yourself, but about creating a system others can rely on."

result4 = evaluate_answer(test_q4, test_a4, domain="HR")
print(result4)

{
  "content_score": 9,
  "strengths": "The candidate clearly demonstrates proactive leadership using the STAR framework, highlighting specific coordination tactics and leading by example. The inclusion of measurable results and a thoughtful reflection on leadership adds strong credibility.",
  "missing_points": "Providing brief context on the nature of the project and why the team was originally behind schedule would make the scenario even more compelling.",
  "structure_feedback": "The response is concise, well-paced, and seamlessly transitions from problem to action, outcome, and key takeaway."
}
